# 외국인 소비 히트맵 & 잠재 관광 상권 발굴

전처리 → 문제 제기 → 프로파일 → k 선정 → 유형 해석 → 성장률 → 후보 도출

분석 로직은 모두 `src/` 모듈에 있고, 이 노트북은 단계별 중간 결과를 눈으로 확인하는 용도다.
전체를 한 번에 돌리려면 `python src/run_all.py`.

> 원본 데이터는 저장소에 포함하지 않는다. `data/ABP_CONTEST_DATA.csv`를 로컬에 두고 실행한다.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from config import FEATURE_COLS, TYPE_URBAN, ensure_dirs, setup_font

ensure_dirs()
print("font:", setup_font())
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. 전처리

업종명 공백 제거, 갈비전문점·한정식 → 일반한식 병합, `시도+시군구` 키 생성.

In [ ]:
import preprocess as pp

df = pp.load()
rep = pp.sanity_report(df)
print(f"{rep['rows']:,}행 / {rep['regions']}개 시군구 / 결측 {rep['na_count']} / 중복 {rep['dup_count']}")
print(f"전체 {rep['total_amt']/1e12:.3f}조, 외국인 {rep['foreign_amt']/1e8:,.0f}억 ({rep['foreign_share']*100:.2f}%)")
pd.Series(rep["monthly_amt"]).div(1e12).round(3)

## 2. 문제 제기

외국인 소비 금액 상위는 제주를 빼면 산업단지·근로자 밀집지다.
**규모만으로 관광 상권을 고르면 틀린다.**

In [ ]:
import features as ft
import viz

feat = ft.build_features(df)
viz.fig1_top15(feat)

feat.nlargest(10, "외국인금액")[["외국인금액", "외국인비중", "장보기비중", "카페양식비중"]]

## 3. 분석 대상 + k 선정

외국인 10억 이상 199곳, 제주 2곳은 학습에서 제외하고 벤치마크로만.

In [ ]:
import cluster as cl

target, train, bench = ft.split_targets(feat)
print(f"대상 {len(target)} / 학습 {len(train)} / 벤치마크 {len(bench)}")

scaler, X = cl.scale(train)
scores = cl.select_k(X)
k = cl.best_k(scores)
print(f"채택 k={k}, 시드 10회 평균 ARI={cl.stability(X, k):.3f}")
viz.fig2_select_k(scores, k)
scores

## 4. 군집 + 유형 명명

카페·양식 최고 → 도심 방문형, 남은 군 중 장보기 최고 → 산업단지 근로형, 나머지 → 중장년 정주형.

In [ ]:
scaler, X, km = cl.fit(train, k)
names = cl.name_clusters(train, km.labels_)
train["유형"] = pd.Series(km.labels_, index=train.index).map(names)

prof = train.groupby("유형")[FEATURE_COLS].mean()
prof_z = (prof - train[FEATURE_COLS].mean()) / train[FEATURE_COLS].std()
viz.fig3_profile_heatmap(prof_z)
prof.round(3)

## 5. 제주는 어디에 있나

PCA 2차원은 분산의 64%만 담는다. 제주의 이질성은 9차원 표준화 거리로 판단한다.

In [ ]:
dist = cl.distance_to_centers(scaler, km, pd.concat([train[FEATURE_COLS], bench[FEATURE_COLS]]))
nearest = dist.min(axis=1)
print("일반 지역 최근접 거리 중앙값: %.2f" % nearest.loc[train.index].median())
print(nearest.loc[bench.index].round(2))

viz.fig4_pca(X, train["유형"], scaler.transform(bench[FEATURE_COLS]),
             bench.index.tolist(), nearest.loc[bench.index],
             float(nearest.loc[train.index].median()))

## 6. 계절성 보정 성장률

5월에 전체 소비가 튀므로 단순 증감은 쓰지 않는다.

`상대성장률 = (외국인 Q2/Q1) ÷ (내국인 Q2/Q1) − 1`

In [ ]:
import growth as gr

g = gr.relative_growth(df)
viz.fig5_monthly_index(gr.monthly_index(df, train["유형"]))

train.join(g["상대성장률"]).groupby("유형")["상대성장률"].mean().round(4)

## 7. 잠재 상권 후보

점수 = 0.4×유사도 + 0.4×상대성장률 + 0.2×청년비중 (각 백분위).

In [ ]:
import candidates as cd

table = (train[FEATURE_COLS + ["외국인금액", "유형"]]
         .join(dist.add_prefix("거리_"))
         .join(g["상대성장률"]))
urban_cluster = next(c for c, n in names.items() if n == TYPE_URBAN)

cand, cutoff, flagships = cd.build_candidates(table, urban_cluster)
print("대표 상권 제외:", flagships)
print(f"컷오프 {cutoff:.3f}, 후보 {len(cand)}곳")
viz.fig6_candidates(cand)

cand.head(15)[["점수", "상대성장률", "청년비중", "카페양식비중", "외국인금액"]]

## 다음 단계

1. 외부 데이터 결합 (`docs/external_data.md`) — 등록외국인 수, 외국인 방문자 수
2. 회귀 잔차로 관광 수요 추정
3. 방문 대비 소비 격차로 최종 TOP 5 확정